# 🥉 Bronze — Raw Ingestion com Delta Lake

## Features implementadas

| # | Feature | Descrição |
|---|---------|----------|
| 1 | `TBLPROPERTIES` | CDF, autoOptimize, autoCompact, retenção |
| 2 | `PARTITIONED BY` | Data skipping por `event_date` |
| 3 | `MERGE (UPSERT)` | Ingestão incremental idempotente |
| 4 | **Time Travel** | Query por versão e timestamp |
| 5 | `DESCRIBE HISTORY` | Auditoria completa das operações Delta |
| 6 | `RESTORE TABLE` | Reverter para versão anterior |
| 7 | **Auto Loader** | `cloudFiles` — ingestão incremental de arquivos |
| 8 | **Schema Evolution** | Aceita novas colunas sem quebrar o pipeline |
| 9 | **Shallow / Deep Clone** | Clones rápidos para dev/test |
| 10 | **Liquid Clustering** | Alternativa moderna ao Z-ORDER + PARTITIONED BY |

## 0. Configuração

In [ ]:
from pyspark.sql import functions as F, types as T
from delta.tables import DeltaTable

CATALOG = 'workspace'
SCHEMA  = 'medallion_demo'
N_ROWS  = 5_000_000

spark.sql(f'CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}')
spark.sql(f'USE CATALOG {CATALOG}')
spark.sql(f'USE SCHEMA {SCHEMA}')
print(f'✅ Usando {CATALOG}.{SCHEMA} | {N_ROWS:,} linhas')

## 1. TBLPROPERTIES + PARTITIONED BY

- `delta.enableChangeDataFeed` → Silver lê apenas mudanças (CDF)
- `delta.autoOptimize.optimizeWrite` → compacta na escrita
- `delta.autoOptimize.autoCompact` → compacta em background
- `PARTITIONED BY (event_date)` → data skipping físico

In [ ]:
spark.sql('''
    CREATE TABLE IF NOT EXISTS bronze_sales (
        transaction_id   STRING,
        customer_id      INT,
        product_id       INT,
        category         STRING,
        country          STRING,
        payment_method   STRING,
        unit_price       DOUBLE,
        quantity         INT,
        event_ts         STRING,
        email            STRING,
        _ingest_ts       TIMESTAMP,
        event_date       STRING
    )
    USING DELTA
    PARTITIONED BY (event_date)
    TBLPROPERTIES (
        "delta.enableChangeDataFeed"         = "true",
        "delta.autoOptimize.optimizeWrite"   = "true",
        "delta.autoOptimize.autoCompact"     = "true",
        "delta.deletedFileRetentionDuration" = "interval 7 days",
        "delta.logRetentionDuration"         = "interval 30 days"
    )
''')
print('✅ Tabela bronze_sales criada (ou já existia)')

## 2. 🥉 Geração de dados Bronze — Carga inicial (Lote 1)

Dados crus com sujeira intencional: duplicatas (~1%), preços inválidos (~3%),
e-mails nulos (~2%), categorias com caixa inconsistente.

In [ ]:
categorias = ['Eletronicos', 'Roupas', 'Alimentos', 'Livros',
              'Casa', 'Brinquedos', 'Esportes', 'Beleza']
paises     = ['BR', 'US', 'AR', 'CL', 'MX', 'PT']
pagamentos = ['credit_card', 'debit_card', 'pix', 'boleto', 'paypal']

def pick(col_rand, valores):
    arr = F.array(*[F.lit(v) for v in valores])
    idx = (F.floor(col_rand * F.lit(len(valores))) + 1).cast('int')
    return F.element_at(arr, idx)

base = spark.range(N_ROWS).withColumnRenamed('id', 'row_id')

bronze_df = (
    base
    .withColumn('transaction_id', F.concat(F.lit('TX-'), F.col('row_id').cast('string')))
    .withColumn('customer_id', (F.floor(F.rand(seed=11) * 200_000) + 1).cast('int'))
    .withColumn('product_id',  (F.floor(F.rand(seed=22) * 5_000) + 1).cast('int'))
    .withColumn('_cat', pick(F.rand(seed=33), categorias))
    .withColumn('category',
        F.when(F.rand(seed=34) < 0.33, F.upper(F.col('_cat')))
         .when(F.rand(seed=34) < 0.66, F.lower(F.col('_cat')))
         .otherwise(F.col('_cat')))
    .withColumn('country', pick(F.rand(seed=44), paises))
    .withColumn('payment_method', pick(F.rand(seed=55), pagamentos))
    .withColumn('unit_price',
        F.when(F.rand(seed=66) < 0.03, F.round(F.rand(seed=67) * -50, 2))
         .otherwise(F.round(F.rand(seed=68) * 990 + 10, 2)))
    .withColumn('quantity', (F.floor(F.rand(seed=77) * 5) + 1).cast('int'))
    .withColumn('event_ts',
        F.from_unixtime(
            F.lit(1704067200) + (F.rand(seed=88) * 31_536_000).cast('long')
        ))
    .withColumn('event_date', F.substring('event_ts', 1, 10))
    .withColumn('email',
        F.when(F.rand(seed=99) < 0.02, F.lit(None))
         .otherwise(F.concat(F.lit('user'), F.col('customer_id').cast('string'), F.lit('@mail.com'))))
    .withColumn('_ingest_ts', F.current_timestamp())
    .drop('_cat', 'row_id')
)

dups = bronze_df.sample(fraction=0.01, seed=123)
bronze_df = bronze_df.unionByName(dups)

(bronze_df.write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .partitionBy('event_date')
    .saveAsTable('bronze_sales'))

v0_count = spark.table('bronze_sales').count()
print(f'✅ Bronze v0 gravada: {v0_count:,} linhas')

## 3. MERGE — Ingestão incremental (Lote 2)

Garante **idempotência**: retransmissões não geram duplicatas.

```sql
MERGE INTO target USING source ON target.transaction_id = source.transaction_id
WHEN MATCHED     THEN UPDATE SET *   -- retransmissão
WHEN NOT MATCHED THEN INSERT *       -- novo registro
```

In [ ]:
batch2_new = (
    spark.range(N_ROWS, N_ROWS + 100_000)
    .withColumnRenamed('id', 'row_id')
    .withColumn('transaction_id', F.concat(F.lit('TX-'), F.col('row_id').cast('string')))
    .withColumn('customer_id', (F.floor(F.rand(seed=11) * 200_000) + 1).cast('int'))
    .withColumn('product_id',  (F.floor(F.rand(seed=22) * 5_000) + 1).cast('int'))
    .withColumn('category',    pick(F.rand(seed=33), categorias))
    .withColumn('country',     pick(F.rand(seed=44), paises))
    .withColumn('payment_method', pick(F.rand(seed=55), pagamentos))
    .withColumn('unit_price',  F.round(F.rand(seed=68) * 990 + 10, 2))
    .withColumn('quantity',    (F.floor(F.rand(seed=77) * 5) + 1).cast('int'))
    .withColumn('event_ts',
        F.from_unixtime(F.lit(1735689600) + (F.rand(seed=88) * 86_400).cast('long')))
    .withColumn('event_date',  F.substring('event_ts', 1, 10))
    .withColumn('email', F.concat(F.lit('user'), F.col('customer_id').cast('string'), F.lit('@mail.com')))
    .withColumn('_ingest_ts', F.current_timestamp())
    .drop('row_id')
)

retransmissions = (
    spark.table('bronze_sales').sample(fraction=0.001, seed=999)
    .withColumn('_ingest_ts', F.current_timestamp())
)
batch2 = batch2_new.unionByName(retransmissions)

target = DeltaTable.forName(spark, 'bronze_sales')
(target.alias('tgt')
    .merge(batch2.alias('src'), 'tgt.transaction_id = src.transaction_id')
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute())

v1_count = spark.table('bronze_sales').count()
print(f'✅ MERGE concluído: {v0_count:,} → {v1_count:,} (+{v1_count - v0_count:,} linhas)')

## 4. Time Travel — Consulta de versões anteriores

- `VERSION AS OF N` → versão específica do log Delta
- `TIMESTAMP AS OF 'yyyy-MM-dd'` → ponto no tempo
- Útil para auditoria, debug e reprodução de resultados históricos

In [ ]:
print('=== Versão 0 — Lote 1 ===')
display(spark.sql('SELECT COUNT(*) AS total_v0 FROM bronze_sales VERSION AS OF 0'))

print('\n=== Versão atual ===')
display(spark.sql('SELECT COUNT(*) AS total_atual FROM bronze_sales'))

print('\n=== Novos registros trazidos pelo MERGE (sample) ===')
display(spark.sql('''
    SELECT v1.transaction_id, v1.event_date, v1.country
    FROM   bronze_sales v1
    WHERE  v1.transaction_id NOT IN (
        SELECT transaction_id FROM bronze_sales VERSION AS OF 0
    )
    LIMIT 10
'''))

## 5. DESCRIBE HISTORY + DESCRIBE DETAIL + RESTORE

- `DESCRIBE HISTORY` → log completo de todas as operações com métricas
- `DESCRIBE DETAIL` → tamanho físico, número de arquivos, partições
- `RESTORE` → reverte a tabela para qualquer versão anterior

In [ ]:
print('=== DESCRIBE HISTORY ===')
display(spark.sql('DESCRIBE HISTORY bronze_sales'))

print('\n=== DESCRIBE DETAIL ===')
display(spark.sql('DESCRIBE DETAIL bronze_sales'))

# RESTORE — comentado para não desfazer o MERGE
# spark.sql('RESTORE TABLE bronze_sales TO VERSION AS OF 0')
print('\nℹ️  RESTORE comentado — descomente para reverter para versão 0')

## 6. Auto Loader (`cloudFiles`) — Ingestão incremental de arquivos

O **Auto Loader** processa arquivos novos de armazenamento em nuvem (S3, ADLS, GCS)
de forma incremental, sem re-processar os já ingeridos. Usa **Structured Streaming** internamente.

**Vantagens sobre `spark.read`:**
- Detecta arquivos novos automaticamente (event-driven ou polling)
- Inferência e evolução de schema automáticas
- Checkpoint de progresso — retoma de onde parou após falha
- Suporta JSON, CSV, Parquet, Avro, ORC, texto

> **Pré-requisito**: Unity Catalog Volume ou path em nuvem.
> No Free Edition, usar `/Volumes/<catalog>/<schema>/<volume>/`

In [ ]:
# === Exemplo de Auto Loader (requer Volume no Unity Catalog) ===
#
# Para criar o volume:
# spark.sql('CREATE VOLUME IF NOT EXISTS workspace.medallion_demo.raw_landing')

VOLUME_PATH    = '/Volumes/workspace/medallion_demo/raw_landing'
CHECKPOINT_LOC = f'{VOLUME_PATH}/_checkpoints/bronze_autoloader'
SCHEMA_LOC     = f'{VOLUME_PATH}/_schema/bronze_autoloader'

# Simulação: salva um micro-batch como JSON no Volume para o Auto Loader ler
try:
    sample_batch = (
        spark.table('bronze_sales')
        .sample(fraction=0.001, seed=42)
        .withColumn('_ingest_ts', F.current_timestamp())
    )
    sample_batch.write.mode('overwrite').json(f'{VOLUME_PATH}/incoming/batch_demo/')
    print('✅ Arquivo de demo gravado no Volume')

    # Auto Loader: lê todos os arquivos novos, para quando não há mais (trigger availableNow)
    df_stream = (
        spark.readStream
        .format('cloudFiles')
        .option('cloudFiles.format', 'json')
        .option('cloudFiles.schemaLocation', SCHEMA_LOC)
        .option('cloudFiles.inferColumnTypes', 'true')
        .load(f'{VOLUME_PATH}/incoming/')
    )

    query = (
        df_stream.writeStream
        .format('delta')
        .outputMode('append')
        .option('checkpointLocation', CHECKPOINT_LOC)
        .option('mergeSchema', 'true')
        .trigger(availableNow=True)  # processa tudo disponível e para
        .table('bronze_sales_autoloader')
    )
    query.awaitTermination(timeout=120)
    print(f'✅ Auto Loader concluído: {spark.table("bronze_sales_autoloader").count():,} linhas')

except Exception as e:
    # Volumes requerem Unity Catalog — Free Edition pode ter restrições de path
    print(f'ℹ️  Auto Loader requer Unity Catalog Volume. Erro: {e}')
    print('   Para habilitar: crie um Volume em workspace.medallion_demo.raw_landing')

## 7. Schema Evolution — Novas colunas sem quebrar o pipeline

Em produção, novas colunas chegam com frequência (ex: time de produto adiciona `promo_code`).
Sem tratamento, o Spark lança `AnalysisException: schema mismatch`.

**Opções:**
- `option('mergeSchema', 'true')` → por escrita
- `spark.conf.set('spark.databricks.delta.schema.autoMerge.enabled', 'true')` → global
- `ALTER TABLE ... ADD COLUMNS (...)` → manual

In [ ]:
# Simula chegada de novo campo: promo_code (não existia antes)
batch_with_new_col = (
    spark.table('bronze_sales')
    .sample(fraction=0.001, seed=55)
    .withColumn('promo_code',   F.lit('SUMMER25'))  # coluna nova
    .withColumn('_ingest_ts',   F.current_timestamp())
)

# SEM mergeSchema → falharia com schema mismatch
# COM mergeSchema → aceita e adiciona a coluna nova
(batch_with_new_col.write
    .format('delta')
    .mode('append')
    .option('mergeSchema', 'true')   # ← habilita evolução de schema
    .saveAsTable('bronze_sales'))

print('✅ Schema evoluído: nova coluna promo_code adicionada')
print('   Registros antigos: promo_code = NULL')
print('   Registros novos  : promo_code = SUMMER25')

# Verifica o schema atualizado
new_schema = spark.table('bronze_sales').schema
print(f'\n   Colunas agora: {[f.name for f in new_schema.fields]}')

## 8. Shallow Clone + Deep Clone

| Tipo | Dados copiados | Metadados | Uso recomendado |
|------|--------------|-----------|----------------|
| **SHALLOW CLONE** | ❌ Referencia os arquivos originais | ✅ Sim | Testes rápidos, experimentos, não ocupa espaço |
| **DEEP CLONE** | ✅ Cópia física completa | ✅ Sim | Backup real, migração, ambiente isolado |

Ambos preservam **history, properties, constraints e schema**.

In [ ]:
# Shallow Clone: referencia os mesmos arquivos (quase instantâneo, ~zero espaço)
spark.sql('CREATE OR REPLACE TABLE bronze_sales_dev SHALLOW CLONE bronze_sales')
dev_count = spark.table('bronze_sales_dev').count()
print(f'✅ SHALLOW CLONE criado: {dev_count:,} linhas (sem duplicar dados em disco)')

# Modificações no clone NÃO afetam a tabela original
spark.sql("DELETE FROM bronze_sales_dev WHERE country = 'PT'")
print(f'   Após DELETE no clone: {spark.table("bronze_sales_dev").count():,} linhas')
print(f'   Original inalterado : {spark.table("bronze_sales").count():,} linhas')

# Deep Clone: cópia física real (independente)
spark.sql('CREATE OR REPLACE TABLE bronze_sales_backup DEEP CLONE bronze_sales')
backup_count = spark.table('bronze_sales_backup').count()
print(f'\n✅ DEEP CLONE criado: {backup_count:,} linhas (cópia física independente)')

# Limpa clones demo
# spark.sql('DROP TABLE IF EXISTS bronze_sales_dev')
# spark.sql('DROP TABLE IF EXISTS bronze_sales_backup')

## 9. Liquid Clustering — Alternativa moderna ao Z-ORDER + PARTITIONED BY

**Disponível a partir do Databricks Runtime 13.3 LTS**

O **Liquid Clustering** substitui particionamento estático + Z-ORDER por um sistema de
clustering dinâmico e incremental:

| | Z-ORDER + PARTITIONED BY | Liquid Clustering |
|--|--------------------------|------------------|
| Reorganização | Full table scan | Incremental (só arquivos novos) |
| Mudança de colunas | Reescreve tudo | `ALTER TABLE ... CLUSTER BY` |
| Granularidade | Por partição | Por arquivo |
| Overhead | Alto para tabelas grandes | Baixo |

**Quando usar?** Tabelas que crescem continuamente, com consultas por múltiplas colunas variando com o tempo.

In [ ]:
# Cria tabela com Liquid Clustering (DBR 13.3+)
spark.sql('''
    CREATE OR REPLACE TABLE bronze_sales_liquid
    CLUSTER BY (country, event_date)
    TBLPROPERTIES (
        "delta.enableChangeDataFeed" = "true"
    )
    AS SELECT * FROM bronze_sales
''')
print('✅ Tabela com Liquid Clustering criada')

# OPTIMIZE em tabela com Liquid Clustering aplica o clustering incrementalmente
spark.sql('OPTIMIZE bronze_sales_liquid')
print('✅ Clustering incremental executado com OPTIMIZE')

# Mudar colunas de clustering sem reescrever tudo
spark.sql('ALTER TABLE bronze_sales_liquid CLUSTER BY (country, category)')
print('✅ Colunas de clustering alteradas para (country, category)')
print('   Próximo OPTIMIZE aplicará o novo clustering incrementalmente')

# Verifica as propriedades de clustering
display(spark.sql('DESCRIBE DETAIL bronze_sales_liquid').select('name', 'clusteringColumns', 'numFiles'))